# Proyecto ML – Infracciones de Tráfico (Alto Riesgo)

Notebook principal: carga de datos, definición del target, entrenamiento y guardado del modelo.

## 1. Carga y preparación de datos

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pathlib import Path

PATH_DATA = Path("src/data_sample")
PATH_MODELS = Path("src/models")
PATH_MODELS.mkdir(parents=True, exist_ok=True)

# Cargar datos (colocar dataset_definitivo.csv o muestra en src/data_sample/)
df = pd.read_csv(PATH_DATA / "dataset_definitivo.csv")
df = df.dropna(subset=["NUM_INFRACCIONES", "CUANTIA"]).copy()

## 2. Definición del target (alto riesgo) y split

In [ ]:
X_full = df.copy()
X_train_raw, X_test_raw = train_test_split(X_full, test_size=0.2, random_state=42, shuffle=True)

p80_infracciones = X_train_raw["NUM_INFRACCIONES"].quantile(0.80)
p80_cuantia = X_train_raw["CUANTIA"].quantile(0.80)

def crear_target_alto_riesgo(df_in, thr_infr, thr_cuant):
    return np.where(
        (df_in["NUM_INFRACCIONES"] >= thr_infr) | (df_in["CUANTIA"] >= thr_cuant),
        1, 0
    )

y_train = crear_target_alto_riesgo(X_train_raw, p80_infracciones, p80_cuantia)
y_test = crear_target_alto_riesgo(X_test_raw, p80_infracciones, p80_cuantia)

cols_leak = ["NUM_INFRACCIONES", "CUANTIA"]
X_train = X_train_raw.drop(columns=cols_leak).copy()
X_test = X_test_raw.drop(columns=cols_leak).copy()

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Distribución train:", pd.Series(y_train).value_counts(normalize=True))

## 3. Preprocesado y modelo (ejemplo: Logistic Regression)

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Identificar columnas numéricas y categóricas (ajustar según tu dataset)
num_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

preprocessor = ColumnTransformer(
    [
        ("num", StandardScaler(), num_cols) if num_cols else ("num", "passthrough", []),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols) if cat_cols else ("cat", "passthrough", []),
    ],
    remainder="drop"
)

pipe = Pipeline([
    ("pre", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")),
])
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred), "F1:", f1_score(y_test, y_pred, average="macro"))

## 4. Guardado del modelo

In [ ]:
import joblib

joblib.dump(pipe, PATH_MODELS / "modelo_final.joblib")
print("Modelo guardado en", PATH_MODELS / "modelo_final.joblib")